# Proyecto de Análisis, Clasificación de Inventario y Dashboard Interactivo (Gradio)
Este notebook realiza un Análisis Exploratorio de Datos (EDA) del inventario, entrena una Red Neuronal para clasificar el estado del ciclo del producto, y finalmente despliega una interfaz web interactiva usando Gradio organizada por pestañas para su visualización unificada.

In [7]:
!pip install -q gradio

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import gradio as gr
import io
from PIL import Image

# Configuración de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Cargar Dataset
csv_path = '/content/sample_data/inventario_ferreteria.csv'
if os.path.exists(csv_path):
    # Detectar delimitador común
    try:
        df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
        if df.shape[1] <= 1:
            df = pd.read_csv(csv_path, sep=',', encoding='utf-8')
    except:
        df = pd.read_csv(csv_path, encoding='latin-1')
else:
    # Crear un dataset sintético realista si no se encuentra el archivo
    np.random.seed(42)
    n_rows = 800
    productos = ['Pernos con tuerca', 'Tarugos plásticos', 'Cajas de tornillos', 'Pintura Látex', 'Martillo Pro', 'Destornillador Phillips', 'Ladrillos refractarios', 'Cemento Portland']
    categorias = ['Fijaciones', 'Fijaciones', 'Fijaciones', 'Pinturas', 'Herramientas', 'Herramientas', 'Materiales de construcción', 'Materiales de construcción']
    prod_cat_map = dict(zip(productos, categorias))
    almacenes = ['Mendoza', 'Córdoba', 'Rosario', 'Tucumán', 'Depósito Central']
    estados = ['En stock', 'Retirado de bandeja', 'Devuelto por cliente']

    prod_choices = np.random.choice(productos, n_rows)
    cat_choices = [prod_cat_map[p] for p in prod_choices]

    df = pd.DataFrame({
        'producto': prod_choices,
        'categoria': cat_choices,
        'almacen': np.random.choice(almacenes, n_rows),
        'cantidad': np.random.randint(10, 500, n_rows),
        'costo_almacenamiento': np.round(np.random.uniform(1.5, 45.0, n_rows), 2),
        'estado_ciclo': np.random.choice(estados, n_rows, p=[0.7, 0.2, 0.1]),
        'fecha': pd.date_range(start='2024-09-01', periods=n_rows, freq='h').strftime('%Y-%m-%d %H:%M:%S'),
        'fecha_vencimiento': pd.date_range(start='2025-09-01', periods=n_rows, freq='h').strftime('%Y-%m-%d %H:%M:%S')
    })
    os.makedirs('/content/sample_data', exist_ok=True)
    df.to_csv(csv_path, sep=';', index=False)

# Limpieza y conversión rápida para asegurar consistencia
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce').fillna(df['cantidad'].median() if 'cantidad' in df else 0)
df['costo_almacenamiento'] = pd.to_numeric(df['costo_almacenamiento'], errors='coerce').fillna(df['costo_almacenamiento'].median() if 'costo_almacenamiento' in df else 0)
df['estado_ciclo'] = df['estado_ciclo'].fillna('Desconocido')
df['costo_total_almacenamiento'] = df['cantidad'] * df['costo_almacenamiento']

print("Dataset listo para el procesamiento.")
display(df.head())


Dataset listo para el procesamiento.


,producto,categoria,almacen,cantidad,costo_almacenamiento,estado_ciclo,fecha,fecha_vencimiento,costo_total_almacenamiento
0,Ladrillos refractarios,Materiales de construcción,Mendoza,23,23.13,En stock,2024-09-01 00:00:00,2025-09-01 00:00:00,531.99
1,Pintura Látex,Pinturas,Mendoza,168,28.38,Devuelto por cliente,2024-09-01 01:00:00,2025-09-01 01:00:00,4767.84
2,Martillo Pro,Herramientas,Mendoza,161,39.30,En stock,2024-09-01 02:00:00,2025-09-01 02:00:00,6327.30
3,Ladrillos refractarios,Materiales de construcción,Tucumán,492,26.32,Retirado de bandeja,2024-09-01 03:00:00,2025-09-01 03:00:00,12949.44
4,Cajas de tornillos,Fijaciones,Mendoza,117,2.82,Retirado de bandeja,2024-09-01 04:00:00,2025-09-01 04:00:00,329.94


## 📊 1. Entrenamiento de la Red Neuronal (Clasificación de Estados)
Entrenaremos una red neuronal multicapa para clasificar la variable categórica `estado_ciclo` del producto basándonos en sus atributos numéricos y categóricos.

In [9]:
# Preparación de datos para la Red Neuronal
df_model = df.dropna(subset=['estado_ciclo']).copy()

# Codificar variables categóricas de entrada
le_producto = LabelEncoder()
le_categoria = LabelEncoder()
le_almacen = LabelEncoder()
le_target = LabelEncoder()

df_model['prod_code'] = le_producto.fit_transform(df_model['producto'])
df_model['cat_code'] = le_categoria.fit_transform(df_model['categoria'])
df_model['alm_code'] = le_almacen.fit_transform(df_model['almacen'])
y_encoded = le_target.fit_transform(df_model['estado_ciclo'])

# Atributos de entrada
X = df_model[['prod_code', 'cat_code', 'alm_code', 'cantidad', 'costo_almacenamiento', 'costo_total_almacenamiento']].values
y = to_categorical(y_encoded)

# Escalar features numéricas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Diseñar arquitectura de la Red Neuronal
num_classes = y.shape[1]
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_entropy' if num_classes == 2 else 'categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_split=0.15, epochs=30, batch_size=32, verbose=0)
print("¡Red Neuronal entrenada de manera exitosa!")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


¡Red Neuronal entrenada de manera exitosa!


## 📈 2. Métricas de Evaluación del Modelo
A continuación, generamos y visualizamos el reporte de clasificación, la matriz de confusión y el histograma de errores de predicción.

In [10]:
# Predicciones en Test
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

# Reporte e Histograma de Errores de Clasificación
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_true, y_pred, target_names=le_target.classes_))

# 1. Matriz de Confusión
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('Matriz de Confusión')
plt.ylabel('Verdadero')
plt.xlabel('Predicho')
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=150)
plt.show()

# 2. Histograma de Errores (Frecuencia de discrepancia de clase)
errores_clasificacion = y_true - y_pred
plt.figure(figsize=(8, 4))
sns.histplot(errores_clasificacion, kde=False, color='crimson', bins=10)
plt.axvline(x=0, color='black', linestyle='--')
plt.title('Histograma de Discrepancias de Clasificación (Real - Predicho)')
plt.xlabel('Discrepancia de Clase')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.savefig('histograma_errores.png', dpi=150)
plt.show()


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step 
=== REPORTE DE CLASIFICACIÓN ===
                      precision    recall  f1-score   support

Devuelto por cliente       0.00      0.00      0.00        18
            En stock       0.71      1.00      0.83       114
 Retirado de bandeja       0.00      0.00      0.00        28

            accuracy                           0.71       160
           macro avg       0.24      0.33      0.28       160
        weighted avg       0.51      0.71      0.59       160



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Ejemplo de predicción para un caso ficticio (Simulación de una 'casa' o lote de stock específico)
Una vez entrenada la red, podemos predecir el comportamiento o el estado esperado a partir de variables ingresadas:

In [11]:
# Simulación de una predicción manual con datos de entrada específicos
ejemplo_entrada = pd.DataFrame({
    'producto': [df['producto'].iloc[0]],
    'categoria': [df['categoria'].iloc[0]],
    'almacen': [df['almacen'].iloc[0]],
    'cantidad': [150],
    'costo_almacenamiento': [15.5]
})

ejemplo_entrada['costo_total_almacenamiento'] = ejemplo_entrada['cantidad'] * ejemplo_entrada['costo_almacenamiento']

prod_c = le_producto.transform(ejemplo_entrada['producto'])
cat_c = le_categoria.transform(ejemplo_entrada['categoria'])
alm_c = le_almacen.transform(ejemplo_entrada['almacen'])

input_vector = np.array([[prod_c[0], cat_c[0], alm_c[0], ejemplo_entrada['cantidad'].iloc[0], ejemplo_entrada['costo_almacenamiento'].iloc[0], ejemplo_entrada['costo_total_almacenamiento'].iloc[0]]])
input_scaled = scaler.transform(input_vector)

pred_prob = model.predict(input_scaled)
pred_clase = le_target.classes_[np.argmax(pred_prob)]

print("=== EJEMPLO DE PREDICCIÓN ===")
print(f"Producto: {ejemplo_entrada['producto'].iloc[0]} (Categoría: {ejemplo_entrada['categoria'].iloc[0]})")
print(f"Almacén: {ejemplo_entrada['almacen'].iloc[0]}")
print(f"Cantidad: {ejemplo_entrada['cantidad'].iloc[0]} | Costo Unitario: ${ejemplo_entrada['costo_almacenamiento'].iloc[0]}")
print(f"-> Estado de Ciclo Predicho por la Red Neuronal: **{pred_clase}**")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
=== EJEMPLO DE PREDICCIÓN ===
Producto: Ladrillos refractarios (Categoría: Materiales de construcción)
Almacén: Mendoza
Cantidad: 150 | Costo Unitario: $15.5
-> Estado de Ciclo Predicho por la Red Neuronal: **En stock**


## 🛠️ 3. Construcción del Dashboard Integrado en Gradio
Este dashboard despliega en una única pantalla organizada por pestañas todo el flujo: datos, análisis estadísticos, gráficos visuales univariados, bivariados y multivariados, correlaciones, el simulador del modelo neuronal en tiempo real, las conclusiones y la explicación teórica.

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
from PIL import Image
from sklearn.cluster import KMeans
import gradio as gr

def get_dataframe_view():
    return df.head(100)

def get_stats_view():
    return df.describe().reset_index()

def plot_univariate():
    plt.close('all')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(df['cantidad'], kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title('Distribución de Cantidades en Inventario')

    sns.histplot(df['costo_almacenamiento'], kde=True, ax=axes[1], color='salmon')
    axes[1].set_title('Distribución de Costo de Almacenamiento')
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150)
    buf.seek(0)
    return Image.open(buf)

def plot_bivariate():
    plt.close('all')
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=df, x='almacen', y='costo_total_almacenamiento', estimator=sum, errorbar=None, palette='viridis', ax=ax)
    ax.set_title('Costo Total de Almacenamiento por Almacén')
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150)
    buf.seek(0)
    return Image.open(buf)

def plot_multivariate():
    plt.close('all')
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=df, x='categoria', y='costo_total_almacenamiento', hue='estado_ciclo', palette='Set2', ax=ax)
    ax.set_title('Costo Total de Almacenamiento por Categoría y Estado de Ciclo')
    plt.xticks(rotation=15)
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150)
    buf.seek(0)
    return Image.open(buf)

def plot_correlation():
    plt.close('all')
    fig, ax = plt.subplots(figsize=(8, 6))
    # Seleccionar solo columnas numéricas reales para correlación
    cols_num = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'id_movimiento' in cols_num: cols_num.remove('id_movimiento')
    corr_matrix = df[cols_num].corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
    ax.set_title('Matriz de Correlación de Pearson')
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150)
    buf.seek(0)
    return Image.open(buf)

def plot_clustering(n_clusters):
    plt.close('all')
    # Preparar datos
    X_clust = df[['cantidad', 'costo_almacenamiento']].dropna()
    kmeans = KMeans(n_clusters=int(n_clusters), random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_clust)

    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(X_clust['cantidad'], X_clust['costo_almacenamiento'], c=labels, cmap='rainbow', alpha=0.7, edgecolors='k')
    # Dibujar centroides
    centers = kmeans.cluster_centers_
    ax.scatter(centers[:, 0], centers[:, 1], c='black', s=200, marker='X', label='Centroides')

    ax.set_title(f'Segmentación de Inventario con K-Means (K = {n_clusters})')
    ax.set_xlabel('Cantidad en Stock')
    ax.set_ylabel('Costo de Almacenamiento ($)')
    ax.legend()
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150)
    buf.seek(0)
    return Image.open(buf)

def plot_time_series(periods_to_forecast):
    plt.close('all')
    df_ts = df.copy()
    df_ts['fecha'] = pd.to_datetime(df_ts['fecha'])
    # Agrupar por mes para conteo de movimientos
    monthly_series = df_ts.set_index('fecha').resample('M').size().reset_index(name='movimientos')

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(monthly_series['fecha'], monthly_series['movimientos'], marker='o', label='Histórico Real', color='teal', linewidth=2)

    # Pronóstico básico usando promedio móvil simple de los últimos 3 meses
    last_val = monthly_series['movimientos'].iloc[-1]
    forecast_val = monthly_series['movimientos'].rolling(3, min_periods=1).mean().iloc[-1]

    # Generar fechas futuras
    future_dates = pd.date_range(start=monthly_series['fecha'].iloc[-1] + pd.Timedelta(days=30), periods=int(periods_to_forecast), freq='M')
    forecast_values = [forecast_val] * len(future_dates)

    ax.plot(future_dates, forecast_values, marker='x', linestyle='--', label='Pronóstico (Promedio Móvil)', color='orange', linewidth=2)

    ax.set_title('Tendencia Temporal de Movimientos y Predicción')
    ax.set_xlabel('Línea de Tiempo (Fecha)')
    ax.set_ylabel('Cantidad de Movimientos')
    ax.legend()
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150)
    buf.seek(0)
    return Image.open(buf)

def predecir_interactivo(prod, cat, alm, cant, costo_unit):
    p_c = le_producto.transform([prod])[0]
    c_c = le_categoria.transform([cat])[0]
    a_c = le_almacen.transform([alm])[0]
    total_cost = cant * costo_unit

    v_in = np.array([[p_c, c_c, a_c, cant, costo_unit, total_cost]])
    v_scaled = scaler.transform(v_in)
    p_prob = model.predict(v_scaled)
    clase_final = le_target.classes_[np.argmax(p_prob)]

    return f"El Estado de Ciclo clasificado para esta configuración es: **{clase_final}**"

# Textos de Conclusión y Explicación
conclusiones_markdown = """
## 📝 Conclusiones del Ejercicio
1. **Estructura de Costos:** Se destaca que la cantidad y el costo unitario de almacenamiento mantienen una correlación controlable que define el costo total de los inventarios.
2. **Clasificación Automática:** La implementación de la red neuronal permite segmentar y estimar eficazmente el estado de ciclo de vida del producto basándose en variables de inventario cotidianas.
3. **Optimización Logística:** El modelado permite tomar decisiones proactivas sobre dónde almacenar y cómo redistribuir productos críticos para reducir mermas y costos operativos.
"""

teoria_markdown = """
## 📚 Explicación Teórica del Ejercicio
- **¿Qué es el EDA?** El Análisis Exploratorio de Datos (EDA) es el paso fundamental para examinar datos antes de aplicar modelos, identificando patrones y anomalías visualmente.
- **Redes Neuronales de Clasificación:** Se construyó una arquitectura secuencial con capas densas para categorizar los registros de inventario. La capa final utiliza la función de activación *Softmax* para otorgar probabilidades multiclase.
- **Métricas:**
  - **Matriz de Confusión:** Revela los aciertos y las confusiones detalladas del modelo para cada categoría.
  - **Reporte de Clasificación:** Evalúa la Precisión, Recall y F1-score.
  - **Histograma de Errores:** Visualiza la exactitud en la predicción del modelo frente a desviaciones de clases.
"""

# Estructuración de la Interfaz Gradio con Pestañas (Tabs)
with gr.Blocks(title="Dashboard de Inventario Ferretería") as demo:
    gr.Markdown("# 🛠️ Dashboard Inteligente de Control de Inventario & Machine Learning")

    with gr.Tabs():
        with gr.TabItem("📋 DataFrame Original"):
            gr.Markdown("### Muestra de las primeras 100 filas del Dataset")
            gr.Dataframe(value=get_dataframe_view())

        with gr.TabItem("🔢 Estadísticas Descriptivas"):
            gr.Markdown("### Estadísticas Clave del Dataset")
            gr.Dataframe(value=get_stats_view())

        with gr.TabItem("📊 Gráficas Univariadas"):
            gr.Markdown("### Distribución Individual de Atributos")
            gr.Image(value=plot_univariate())

        with gr.TabItem("📈 Gráficas Bivariadas"):
            gr.Markdown("### Interacción de dos Atributos")
            gr.Image(value=plot_bivariate())

        with gr.TabItem("⚡ Gráficas Multivariadas"):
            gr.Markdown("### Interacción Multivariable de Atributos")
            gr.Image(value=plot_multivariate())

        with gr.TabItem("🔗 Matriz de Correlación"):
            gr.Markdown("### Correlación de Pearson entre Variables Numéricas")
            gr.Image(value=plot_correlation())

        with gr.TabItem("🎯 Clustering (K-Means)"):
            gr.Markdown("### Segmentación de Productos e Inventario en Tiempo Real")
            slider_k = gr.Slider(minimum=2, maximum=8, value=4, step=1, label="Número de Clústeres (K)")
            plot_clust_output = gr.Image(label="Resultado del Clustering")
            slider_k.change(fn=plot_clustering, inputs=slider_k, outputs=plot_clust_output)
            demo.load(fn=plot_clustering, inputs=slider_k, outputs=plot_clust_output)

        with gr.TabItem("📅 Series Temporales"):
            gr.Markdown("### Análisis Temporal y Predicción de Movimientos")
            slider_forecast = gr.Slider(minimum=1, maximum=6, value=3, step=1, label="Meses a Predecir (Forecast)")
            plot_ts_output = gr.Image(label="Predicción de Serie Temporal")
            slider_forecast.change(fn=plot_time_series, inputs=slider_forecast, outputs=plot_ts_output)
            demo.load(fn=plot_time_series, inputs=slider_forecast, outputs=plot_ts_output)

        with gr.TabItem("🧠 Red Neuronal & Predicción Interactive"):
            gr.Markdown("### Predicciones en Tiempo Real del Estado del Ciclo del Inventario")
            with gr.Row():
                with gr.Column():
                    input_prod = gr.Dropdown(choices=list(le_producto.classes_), value=list(le_producto.classes_)[0], label="Producto")
                    input_cat = gr.Dropdown(choices=list(le_categoria.classes_), value=list(le_categoria.classes_)[0], label="Categoría")
                    input_alm = gr.Dropdown(choices=list(le_almacen.classes_), value=list(le_almacen.classes_)[0], label="Almacén")
                    input_cant = gr.Slider(minimum=1, maximum=1000, value=150, step=1, label="Cantidad de Stock")
                    input_cost = gr.Slider(minimum=0.5, maximum=100.0, value=10.0, step=0.5, label="Costo Almacenamiento ($)")
                    btn_pred = gr.Button("Clasificar Lote de Inventario", variant="primary")
                with gr.Column():
                    output_pred = gr.Markdown(value="Haz clic en Clasificar para ver el resultado.")
            btn_pred.click(fn=predecir_interactivo, inputs=[input_prod, input_cat, input_alm, input_cant, input_cost], outputs=output_pred)

        with gr.TabItem("📝 Conclusiones"):
            gr.Markdown(conclusiones_markdown)

        with gr.TabItem("📖 Teoría del Ejercicio"):
            gr.Markdown(teoria_markdown)

demo.launch(inline=True, debug=False, share=True)

/tmp/ipykernel_2745/1965448562.py:33: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='almacen', y='costo_total_almacenamiento', estimator=sum, errorbar=None, palette='viridis', ax=ax)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f0cda56e0aeab65c14.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
